In [12]:
import os
import sys
import re
import json
import random
import warnings
import subprocess
import importlib
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Папка артефактов
os.makedirs("artifacts", exist_ok=True)

def safe_ensure_package(package_name: str, import_name: Optional[str] = None, auto_install: bool = False) -> bool:
    """Проверяет пакет. Если auto_install=True и пакета нет, ставит через pip."""
    target = import_name or package_name
    try:
        importlib.import_module(target)
        return True
    except ImportError:
        if auto_install:
            print(f" Пакет '{package_name}' не найден. Устанавливаем...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
                importlib.import_module(target)
                return True
            except Exception as e:
                print(f" Ошибка установки '{package_name}': {e}")
                return False
        else:
            print(f" Пакет '{package_name}' отсутствует. Для воспроизводимости установите вручную.")
            return False

# Разрешаем автоустановку только при явном флаге
ALLOW_AUTO_INSTALL = os.environ.get("ALLOW_AUTO_INSTALL", "0") == "1"

FAISS_READY = safe_ensure_package("faiss-cpu", "faiss", auto_install=ALLOW_AUTO_INSTALL)
try:
    import faiss
except Exception:
    faiss = None

ST_READY = safe_ensure_package("sentence-transformers", "sentence_transformers", auto_install=ALLOW_AUTO_INSTALL)

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass

set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print(f" Устройство: {DEVICE}")
print(f" FAISS: {'доступен' if FAISS_READY else 'fallback на TF-IDF'}")
print(f" sentence-transformers: {'доступен' if ST_READY else 'fallback на TF-IDF'}")

 Устройство: cpu
 FAISS: доступен
 sentence-transformers: доступен


In [ ]:
documents = [
    {"doc_id": "doc_01", "title": "Эмбеддинги текстов", "text": "Эмбеддинг – это векторное представление текста. Семантически похожие фразы оказываются близко в векторном пространстве. Это позволяет искать документы не только по совпадению слов, но и по смыслу запроса."},
    {"doc_id": "doc_02", "title": "FAISS и поиск ближайших соседей", "text": "FAISS – библиотека для быстрого поиска ближайших соседей по векторам. Она полезна, когда в базе знаний много чанков и нужно быстро находить top-k наиболее похожих фрагментов."},
    {"doc_id": "doc_03", "title": "Чанкинг и overlap", "text": "Чанкинг разбивает длинный документ на более короткие фрагменты. Если чанк слишком большой, retrieval может возвращать слишком общий контекст. Если чанк слишком маленький, смысл распадается. Overlap помогает не потерять информацию на границах соседних фрагментов."},
    {"doc_id": "doc_04", "title": "Оценка качества retrieval", "text": "Качество retrieval нельзя оценивать только визуально. Обычно используют hit@k, recall@k и MRR. Эти метрики помогают понять, нашел ли retriever релевантный документ и насколько высоко он оказался в выдаче."},
    {"doc_id": "doc_05", "title": "Обновление базы знаний", "text": "После добавления новых документов база знаний должна быть переиндексирована. Иначе retriever не увидит новые фрагменты. В production-системах обновление индекса может быть периодическим или событийным."},
    {"doc_id": "doc_06", "title": "Галлюцинации в RAG", "text": "RAG снижает риск галлюцинаций, но не устраняет его полностью. Если retrieval вернул нерелевантные фрагменты или генератор исказил найденный факт, итоговый ответ все равно будет ошибочным."},
    {"doc_id": "doc_07", "title": "Промпт с контекстом", "text": "В RAG-сценарии важно явно передавать модели найденный контекст и ограничивать ответ этим контекстом. Полезно просить систему отвечать только на основании источников и возвращать указание на использованные фрагменты."},
    {"doc_id": "doc_08", "title": "Метаданные и фильтрация", "text": "Помимо текста, retrieval может учитывать метаданные: тип документа, дату, автора, подразделение или теги. Фильтрация по метаданным уменьшает область поиска и повышает точность извлечения."},
    {"doc_id": "doc_09", "title": "Реранжирование результатов", "text": "После первичного retrieval можно применять реранжирование. Сначала быстрый retriever извлекает кандидатов, а затем более точная модель пересортировывает их по полезности для ответа."},
    {"doc_id": "doc_10", "title": "Гибридный поиск", "text": "Гибридный поиск сочетает dense retrieval и классический лексический поиск. Он полезен, когда часть запросов требует смыслового сходства, а часть – точного совпадения терминов или аббревиатур."}
]

docs_df = pd.DataFrame(documents)
print(f" Документов: {len(docs_df)}")
display(docs_df.head())
print(" Тема: Retrieval и RAG. Датасет содержит чёткие определения концептов, что идеально для отладки точности извлечения и оценки mini-RAG.")

: 

In [ ]:
def chunk_text(text: str, chunk_size: int = 24, overlap: int = 6) -> List[str]:
    words = text.split()
    if chunk_size <= 0: raise ValueError("chunk_size > 0")
    if overlap >= chunk_size: raise ValueError("overlap < chunk_size")
    
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words: continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words): break
    return chunks

print("🔪 Пример чанкинга (doc_03):")
for i, c in enumerate(chunk_text(documents[2]["text"], 24, 6), 1):
    print(f"[{i}] {c}")

: 

In [ ]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray: raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray: raise NotImplementedError

class TfidfBackend(EmbeddingBackend):
    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        m = self.vectorizer.fit_transform(texts).astype(np.float32).toarray()
        return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-12)
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        m = self.vectorizer.transform(texts).astype(np.float32).toarray()
        return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-12)

class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"ST: {model_name.split('/')[-1]}"
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    if ST_READY:
        try: return SentenceTransformersBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device)
        except: return TfidfBackend()
    return TfidfBackend()

@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object

def build_retriever(docs: List[Dict], chunk_size: int = 24, overlap: int = 6, device: str = "cpu") -> RetrieverArtifacts:
    rows = []
    for doc in docs:
        for idx, chunk in enumerate(chunk_text(doc["text"], chunk_size, overlap), 1):
            rows.append({"doc_id": doc["doc_id"], "title": doc["title"], 
                         "chunk_id": f'{doc["doc_id"]}_chunk_{idx:02d}', "chunk_text": chunk})
    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device)
    vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())
    
    if FAISS_READY:
        idx = faiss.IndexFlatIP(vectors.shape[1])
        idx.add(vectors)
    else:
        idx = vectors
        
    return RetrieverArtifacts(backend.backend_name, chunks_df, vectors, backend, idx)

def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    q_vec = artifacts.backend.encode_queries([query]).astype(np.float32)
    if FAISS_READY:
        scores, indices = artifacts.index.search(q_vec, top_k)
        scores, indices = scores[0], indices[0]
    else:
        sims = (artifacts.chunk_vectors @ q_vec.T).reshape(-1)
        indices = np.argsort(-sims)[:top_k]
        scores = sims[indices]
    
    res = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    res["rank"] = np.arange(1, len(res)+1)
    res["score"] = scores
    return res[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

artifacts = build_retriever(documents, chunk_size=24, overlap=6, device=DEVICE)
print(f" Backend: {artifacts.backend_name} | Чанков: {len(artifacts.chunks_df)}")

: 

In [ ]:
control_queries = [
    {"query": "Что такое эмбеддинг?", "expected": ["doc_01"]},
    {"query": "Для чего используется FAISS?", "expected": ["doc_02"]},
    {"query": "Зачем нужен overlap при разбиении текста?", "expected": ["doc_03"]},
    {"query": "Какие метрики используются для оценки retrieval?", "expected": ["doc_04"]},
    {"query": "Что происходит с индексом после добавления документов?", "expected": ["doc_05"]},
    {"query": "Полностью ли RAG устраняет галлюцинации?", "expected": ["doc_06"]},
    {"query": "Как правильно формировать промпт с контекстом?", "expected": ["doc_07"]},
    {"query": "Как метаданные помогают в поиске?", "expected": ["doc_08"]},
    {"query": "Зачем нужно реранжирование после первичного поиска?", "expected": ["doc_09"]},
    {"query": "Когда применяют гибридный поиск?", "expected": ["doc_10"]}
]

def evaluate_retrieval(queries: List[Dict], artifacts: RetrieverArtifacts, k: int = 3) -> pd.DataFrame:
    rows = []
    for item in queries:
        res = search_chunks(item["query"], artifacts, top_k=k)
        retrieved = res["doc_id"].tolist()
        expected = item["expected"]
        hit = int(any(doc in retrieved for doc in expected))
        relevant_found = sum(1 for d in retrieved if d in expected)
        recall = relevant_found / len(expected) if expected else 0.0
        first_rank = next((int(r) for r, d in zip(res["rank"], retrieved) if d in expected), None)
        rows.append({"query": item["query"], "expected_source": ", ".join(expected),
                     "retrieved_sources": ", ".join(retrieved), "hit_at_k": hit,
                     "recall_at_k": recall, "rank_of_first_relevant": first_rank})
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(control_queries, artifacts, k=3)
display(eval_df)
print(f"\n Итог: hit@3={eval_df['hit_at_k'].mean():.2f}, recall@3={eval_df['recall_at_k'].mean():.2f}")
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/retrieval_eval.csv")

: 

In [ ]:
artifacts_v2 = build_retriever(documents, chunk_size=48, overlap=10, device=DEVICE)
eval_v2 = evaluate_retrieval(control_queries, artifacts_v2, k=3)

print(" Сравнение chunk_size:")
print(f"24 слова: hit@3={eval_df['hit_at_k'].mean():.2f}, recall@3={eval_df['recall_at_k'].mean():.2f}")
print(f"48 слов:  hit@3={eval_v2['hit_at_k'].mean():.2f}, recall@3={eval_v2['recall_at_k'].mean():.2f}")
print(" Вывод: chunk_size=24 выбран как основной. Меньшие фрагменты точнее попадают в релевантные концепты.")

: 

In [ ]:
new_docs = [
    {"doc_id": "doc_11", "title": "Кэширование векторных запросов", "text": "Для снижения задержек часто используется кэширование результатов retrieval. Повторяющиеся или семантически близкие запросы возвращают сохранённый ответ из кэша, минуя повторный расчет эмбеддингов и поиск по индексу."},
    {"doc_id": "doc_12", "title": "Ограничения контекстного окна", "text": "LLM имеют жесткое ограничение на длину входного контекста. При передаче большого числа чанков в промпт возникает риск усечения (truncation) или потери внимания на важных деталях, что требует стратегического отбора top-k фрагментов."}
]
docs_updated = documents + new_docs
artifacts_updated = build_retriever(docs_updated, chunk_size=24, overlap=6, device=DEVICE)

update_queries = ["Как работает кэширование запросов?", "Что ограничивает длину контекста в LLM?"]
update_rows = []
for q in update_queries:
    before = search_chunks(q, artifacts, top_k=3)["doc_id"].tolist()
    after = search_chunks(q, artifacts_updated, top_k=3)["doc_id"].tolist()
    update_rows.append({"query": q, "before_retrieved_sources": ", ".join(before),
                        "after_retrieved_sources": ", ".join(after), "changed": "Да" if before != after else "Нет"})
update_df = pd.DataFrame(update_rows)
display(update_df)
update_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/retrieval_before_after_update.csv")

: 

In [ ]:
def split_into_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def extract_answer(query: str, context: str, top_n: int = 2) -> str:
    sentences = [s for line in context.split('\n') for s in split_into_sentences(line) if not line.startswith('[Источник')]
    if not sentences: return "Недостаточно контекста."
    vec = TfidfVectorizer(ngram_range=(1,2)).fit_transform([query] + sentences).astype(np.float32).toarray()
    q_vec, s_vecs = vec[0], vec[1:]
    norms_q = np.linalg.norm(q_vec) + 1e-12
    norms_s = np.linalg.norm(s_vecs, axis=1, keepdims=True) + 1e-12
    scores = (s_vecs @ q_vec) / (norms_s.reshape(-1) * norms_q)
    selected, used = [], set()
    for idx in np.argsort(-scores):
        if scores[idx] <= 0: continue
        txt = sentences[idx].lower().strip()
        if txt in used: continue
        used.add(txt)
        selected.append(sentences[idx])
        if len(selected) >= top_n: break
    return " ".join(selected) if selected else "В контексте нет точного ответа."

def mini_rag(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Dict:
    res = search_chunks(query, artifacts, top_k)
    context = "\n".join([f"[Источник: {r.doc_id} | {r.title} | score={r.score:.3f}]\n{r.chunk_text}" for _, r in res.iterrows()])
    answer = extract_answer(query, context)
    return {"question": query, "answer": answer, "retrieved_sources": ", ".join(res["doc_id"].tolist()), "context": context}

test_queries = ["Как влияет overlap на качество чанков?", "Какие метрики используют для оценки retrieval?",
                "Зачем нужно реранжирование?", "Что ограничивает контекстное окно LLM?", "Как работают галлюцинации в RAG?"]
rag_examples = []
for q in test_queries:
    res = mini_rag(q, artifacts_updated)
    rag_examples.append({"question": res["question"], "answer": res["answer"], "retrieved_sources": res["retrieved_sources"]})
    print(f" {res['question']}\n {res['answer']}\n📚 {res['retrieved_sources']}\n")

pd.DataFrame(rag_examples).to_csv("artifacts/rag_examples.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/rag_examples.csv")

: 

In [ ]:
print("📝 Анализ слабых мест:")
for case in rag_examples:
    if len(case["answer"].split()) < 4 or not any(w.lower() in case["answer"].lower() for w in case["question"].split() if len(w)>3):
        print(f"\n--- Вопрос: {case['question']}")
        print(f"Ответ: {case['answer']}")
        print("Комментарий: Проблема extractive-подхода. Генератор ищет лексическое совпадение с вопросом, а не смысловой ответ. Если релевантное предложение не содержит слов из запроса, оно отбрасывается. Также влияет фрагментация: ответ мог быть разделён между чанками.")

: 